# Versão 1: Pré-processamento em Python + Exportação para Excel e Weka (.arff)
**Equipe 11:** Guilherme Santos Paz e Ana Melina  
**Base de Dados:** Nº 22 - Default of Credit Card Clients (ID UCI: 350)  
**Objetivo:** Realizar toda a limpeza, tratamento de códigos não documentados (`EDUCATION` e `PAY`), amostragem estratificada, discretização para Excel e geração dos arquivos `.arff` (Execução 1 e Execução 2) de forma linear.

In [1]:
# Importação das bibliotecas e carregamento da base original
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

In [2]:
# Cria a pasta "results" se ela não existir
os.makedirs('results', exist_ok=True)

In [4]:
# Leitura da base de dados do Excel
df = pd.read_excel('C:/Users/guilh/OneDrive/Área de Trabalho/Projetos/MineraçãodeDados/default of credit card clients.xls', header=1)
print('Formato original da base:', df.shape)
df.head()

Formato original da base: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [5]:
# Ajuste estrutural limpeza de códigos não documentados
# 1. Remove a coluna identificadora ID
df = df.drop(columns=['ID'])

# Renomeia o atributo-classe para facilitar a leitura no Weka
df = df.rename(columns={'default payment next month': 'CLASSE_INADIMPLENCIA'})

# 2. Trata códigos não documentados em EDUCATION (0, 5 e 6 passam a ser 4 = Outros)
print('Valores de EDUCATION antes:', df['EDUCATION'].value_counts().to_dict())
df['EDUCATION'] = df['EDUCATION'].replace([0, 5, 6], 4)
print('Valores de EDUCATION depois:', df['EDUCATION'].value_counts().to_dict())

# 3. Trata código não documentado em MARRIAGE (0 passa a ser 3 = Outros)
df['MARRIAGE'] = df['MARRIAGE'].replace(0, 3)

# 4. Trata códigos não documentados em PAY_0 até PAY_6 (-2 e -1 agrupados com 0 = Pagamento em dia)
colunas_pay = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
for col in colunas_pay:
    df[col] = df[col].replace([-2, -1], 0)

print('Valores de PAY_0 depois:', df['PAY_0'].value_counts().to_dict())

Valores de EDUCATION antes: {2: 14030, 1: 10585, 3: 4917, 5: 280, 4: 123, 6: 51, 0: 14}
Valores de EDUCATION depois: {2: 14030, 1: 10585, 3: 4917, 4: 468}
Valores de PAY_0 depois: {0: 23182, 1: 3688, 2: 2667, 3: 322, 4: 76, 5: 26, 8: 19, 6: 11, 7: 9}


In [6]:
# Conversão de atributos categóricos para texto nominal 
# Isso garante que o Weka entenda que são categorias e não números contínuos
df['SEX'] = df['SEX'].map({1: 'Masculino', 2: 'Feminino'})
df['EDUCATION'] = df['EDUCATION'].map({
    1: 'Pos_Graduacao',
    2: 'Universidade',
    3: 'Ensino_Medio',
    4: 'Outros'
})
df['MARRIAGE'] = df['MARRIAGE'].map({
    1: 'Casado',
    2: 'Solteiro',
    3: 'Outros'
})

for col in colunas_pay:
    df[col] = df[col].map({
        0: 'Em_Dia',
        1: 'Atraso_1mes',
        2: 'Atraso_2mes',
        3: 'Atraso_3mes',
        4: 'Atraso_4mes',
        5: 'Atraso_5mes',
        6: 'Atraso_6mes',
        7: 'Atraso_7mes',
        8: 'Atraso_8mes',
        9: 'Atraso_9mes_ou_mais'
    })

df['CLASSE_INADIMPLENCIA'] = df['CLASSE_INADIMPLENCIA'].map({0: 'Adimplente', 1: 'Inadimplente'})

print('Distribuição percentual da classe na base completa:')
print(df['CLASSE_INADIMPLENCIA'].value_counts(normalize=True) * 100)

Distribuição percentual da classe na base completa:
CLASSE_INADIMPLENCIA
Adimplente      77.88
Inadimplente    22.12
Name: proportion, dtype: float64


In [7]:
# Amostragem Estratificada (Redução de 30.000 para 5.000 instâncias com seed=42)
df_amostra, _ = train_test_split(
    df,
    train_size=5000,
    stratify=df['CLASSE_INADIMPLENCIA'],
    random_state=42
)
df_amostra = df_amostra.reset_index(drop=True)

print('Total de linhas na amostra:', len(df_amostra))
print('Distribuição da classe na amostra (%):')
print(df_amostra['CLASSE_INADIMPLENCIA'].value_counts(normalize=True) * 100)

Total de linhas na amostra: 5000
Distribuição da classe na amostra (%):
CLASSE_INADIMPLENCIA
Adimplente      77.88
Inadimplente    22.12
Name: proportion, dtype: float64


In [12]:
# Exportação do arquivo .ARFF 
# Define o caminho e o nome do arquivo ARFF a ser gerado
caminho_arff_1 = 'C:/Users/guilh/OneDrive/Área de Trabalho/Projetos/MineraçãodeDados/results/base22_execucao1_continua.arff'
arquivo_arff_1 = open(caminho_arff_1, 'w', encoding='utf-8')
arquivo_arff_1.write('@relation credit_card_clients_execucao1\n\n')

# Define as colunas contínuas
colunas_continuas = [
    'LIMIT_BAL', 'AGE',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
]

# Escreve os atributos no arquivo ARFF
# - Variáveis contínuas são declaradas como 'numeric'
# - Variáveis categóricas/nominais (incluindo o alvo) recebem o conjunto explícito de classes {val1,val2,...}
#   para evitar interpretação indevida como contínuas no Weka 
for col in df_amostra.columns:
    if col in colunas_continuas:
        arquivo_arff_1.write(f'@attribute {col} numeric\n')
    else:
        valores_unicos = sorted(df_amostra[col].dropna().unique().tolist())
        lista_str = ','.join(valores_unicos)
        arquivo_arff_1.write(f'@attribute {col} {{{lista_str}}}\n')

# Escrita da seção de dados (@data) linha a linha em formato CSV compatível com Weka
arquivo_arff_1.write('\n@data\n')
for _, linha in df_amostra.iterrows():
    valores_linha = [str(val) for val in linha.values]
    arquivo_arff_1.write(','.join(valores_linha) + '\n')

arquivo_arff_1.close()
print('Arquivo base22_execucao1_continua.arff gerado com sucesso!')

Arquivo base22_execucao1_continua.arff gerado com sucesso!


In [13]:
# Discretização dos Atributos Contínuos e Geração da Planilha Excel
df_discretizado = df_amostra.copy()
tabela_cortes = []

# Discretização de AGE por faixas etárias de negócio
df_discretizado['AGE'] = pd.cut(
    df_amostra['AGE'],
    bins=[0, 29, 45, 60, 120],
    labels=['Jovem_ate_29', 'Adulto_30_45', 'Meia_Idade_46_60', 'Senior_acima_60']
).astype(str)
tabela_cortes.append({'Atributo': 'AGE', 'Critério': 'Faixas Etárias Demográficas', 'Pontos_de_Corte': 'Até 29 | 30 a 45 | 46 a 60 | Acima de 60'})

# Discretização de LIMIT_BAL por quartis (4 faixas de mesmo tamanho)
_, bins_limit = pd.qcut(df_amostra['LIMIT_BAL'], q=4, retbins=True, duplicates='drop')
df_discretizado['LIMIT_BAL'] = pd.qcut(
    df_amostra['LIMIT_BAL'],
    q=4,
    labels=['Limite_Baixo_Q1', 'Limite_Medio_Baixo_Q2', 'Limite_Medio_Alto_Q3', 'Limite_Alto_Q4'],
    duplicates='drop'
).astype(str)
tabela_cortes.append({'Atributo': 'LIMIT_BAL', 'Critério': 'Quartis (Igual Frequência)', 'Pontos_de_Corte': str(np.round(bins_limit, 2).tolist())})

# 3. Discretização de BILL_AMT1..6 e PAY_AMT1..6 por faixas financeiras
colunas_fatura = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
for col in colunas_fatura:
    df_discretizado[col] = pd.cut(
        df_amostra[col],
        bins=[-np.inf, 0, 10000, 50000, np.inf],
        labels=['Sem_Saldo_ou_Negativo', 'Fatura_Baixa_ate_10k', 'Fatura_Media_10k_50k', 'Fatura_Alta_acima_50k']
    ).astype(str)
    tabela_cortes.append({'Atributo': col, 'Critério': 'Faixas de Saldo de Fatura (NT$)', 'Pontos_de_Corte': '<= 0 | 1 a 10.000 | 10.001 a 50.000 | > 50.000'})

colunas_pgto = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
for col in colunas_pgto:
    df_discretizado[col] = pd.cut(
        df_amostra[col],
        bins=[-np.inf, 0, 2000, 10000, np.inf],
        labels=['Sem_Pagamento_0', 'Pgto_Baixo_ate_2k', 'Pgto_Medio_2k_10k', 'Pgto_Alto_acima_10k']
    ).astype(str)
    tabela_cortes.append({'Atributo': col, 'Critério': 'Faixas de Valor Pago (NT$)', 'Pontos_de_Corte': '0 | 1 a 2.000 | 2.001 a 10.000 | > 10.000'})

df_cortes = pd.DataFrame(tabela_cortes)

# Salva a planilha Excel com 3 abas para entregar à professora
caminho_excel = 'C:/Users/guilh/OneDrive/Área de Trabalho/Projetos/MineraçãodeDados/results/Discretizacao_Execucao2_Excel.xlsx'
with pd.ExcelWriter(caminho_excel, engine='openpyxl') as writer:
    df_cortes.to_excel(writer, sheet_name='Pontos_de_Corte', index=False)
    df_amostra.to_excel(writer, sheet_name='Dados_Continuos_Exec1', index=False)
    df_discretizado.to_excel(writer, sheet_name='Dados_Discretizados_Exec2', index=False)

print('Planilha Discretizacao_Execucao2_Equipe11.xlsx gerada com sucesso!')

Planilha Discretizacao_Execucao2_Equipe11.xlsx gerada com sucesso!


In [14]:
# Exportação do arquivo .ARFF da EXECUÇÃO 2 (Todos os Atributos Discretizados)
caminho_arff_2 = 'C:/Users/guilh/OneDrive/Área de Trabalho/Projetos/MineraçãodeDados/results/base22_execucao2_discretizada.arff'
arquivo_arff_2 = open(caminho_arff_2, 'w', encoding='utf-8')
arquivo_arff_2.write('@relation credit_card_clients_execucao2_discretizada\n\n')

for col in df_discretizado.columns:
    valores_unicos = sorted(df_discretizado[col].dropna().unique().tolist())
    lista_str = ','.join(valores_unicos)
    arquivo_arff_2.write(f'@attribute {col} {{{lista_str}}}\n')

arquivo_arff_2.write('\n@data\n')
for _, linha in df_discretizado.iterrows():
    valores_linha = [str(val) for val in linha.values]
    arquivo_arff_2.write(','.join(valores_linha) + '\n')

arquivo_arff_2.close()
print('Arquivo base22_execucao2_discretizada.arff gerado com sucesso!')

Arquivo base22_execucao2_discretizada.arff gerado com sucesso!
